# Análise de Cohen's Kappa

Este notebook calcula o Cohen's Kappa para medir concordância entre dois anotadores:
- **Mendes** (anotador 1)
- **Guilherme** (anotador 2)

Cohen's Kappa avalia a concordância além do acaso, com escala de -1 (discordância total) a 1 (concordância perfeita).

In [3]:
# Importações necessárias
# =======================
# statsmodels: cálculos estatísticos (Cohen's Kappa)
# pandas: manipulação de dados (leitura/escrita Excel)
# numpy: operações numéricas
# os: manipulação de caminhos do sistema de arquivos
# sys: informações do ambiente Python
# sklearn: métricas alternativas de avaliação (confusion_matrix, classification_report)

from sklearn.metrics import cohen_kappa_score
import pandas as pd
import numpy as np
import os
import sys
from sklearn.metrics import confusion_matrix, classification_report

# Diagnóstico de versões
print("="*60)
print("DIAGNÓSTICO DE AMBIENTE")
print("="*60)
print(f"Python version: {sys.version.split()[0]}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
try:
    import statsmodels as sm
    print(f"statsmodels version: {sm.__version__}")
except Exception as e:
    print(f"statsmodels: importação OK (versão não disponível)")
try:
    from sklearn import __version__ as sklearn_version
    from sklearn import __version__
    print(f"scikit-learn version: {__version__}")
except:
    print("scikit-learn: importação OK (versão não disponível)")
print("="*60)

DIAGNÓSTICO DE AMBIENTE
Python version: 3.12.6
pandas version: 3.0.2
numpy version: 2.4.4
statsmodels version: 0.14.6
scikit-learn version: 1.8.0


In [4]:
# Configuração de caminhos e variáveis
# ====================================
# Define diretórios de entrada/saída para os arquivos Excel
# Todos os arquivos devem estar em: data/estatistica_kappa/

CAMINHO_DIRETORIO = os.path.abspath(os.path.join(
    os.path.dirname("sistema-inteligencia-prescritiva-para-industria-farmaceutica"), 
    "..", "..", "data", "estatistica_kappa"
))

# Caminho do arquivo de entrada (contém avaliações de todos os anotadores)
EXCEL_ENTRADA = os.path.join(CAMINHO_DIRETORIO, "noticias_classificadas.xlsx")

# Caminho do arquivo de saída com relatório de Cohen's Kappa
EXCEL_SAIDA = os.path.join(CAMINHO_DIRETORIO, "cohen_kappa_mendes_guilherme.xlsx")

print("\n" + "="*60)
print("CONFIGURAÇÃO DE ARQUIVOS")
print("="*60)
print(f"Diretório de trabalho: {CAMINHO_DIRETORIO}")
print(f"Arquivo de entrada: {EXCEL_ENTRADA}")
print(f"Arquivo de saída: {EXCEL_SAIDA}")
print(f"Arquivo de entrada existe: {os.path.exists(EXCEL_ENTRADA)}")
print("="*60)


CONFIGURAÇÃO DE ARQUIVOS
Diretório de trabalho: c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa
Arquivo de entrada: c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa\noticias_classificadas.xlsx
Arquivo de saída: c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa\cohen_kappa_mendes_guilherme.xlsx
Arquivo de entrada existe: True


In [5]:
# Carregamento e preparação dos dados
# ====================================
# Lê o arquivo Excel com as avaliações de todos os anotadores
# Extrai as colunas de interesse (Mendes e Guilherme)
# Realiza limpeza e padronização dos dados

print("\nCarregando arquivo de entrada...")
try:
    df = pd.read_excel(EXCEL_ENTRADA)
    print(f"✓ Arquivo lido com sucesso")
    print(f"  Total de linhas: {len(df)}")
    print(f"  Total de colunas: {len(df.columns)}")
    print(f"  Colunas disponíveis: {list(df.columns)}")
except FileNotFoundError:
    print(f"✗ Erro: Arquivo não encontrado em {EXCEL_ENTRADA}")
    raise
except Exception as e:
    print(f"✗ Erro ao ler arquivo: {e}")
    raise

# Verifica se as colunas esperadas existem
print("\nValidando colunas esperadas...")
colunas_esperadas = ['classificacao mendes', 'classificacao guilherme']
colunas_encontradas = [c for c in colunas_esperadas if c in df.columns]
colunas_faltantes = [c for c in colunas_esperadas if c not in df.columns]

if colunas_faltantes:
    print(f"✗ Colunas faltantes: {colunas_faltantes}")
    print("\nTentando encontrar colunas com nomes parcialmente iguais...")
    # Lowercase e strip para comparação fuzzy
    df_cols_normalized = {col: col.lower().strip() for col in df.columns}
    for col_norm, col_orig in df_cols_normalized.items():
        if 'mendes' in col_norm and 'classificacao' in col_norm:
            print(f"  Encontrado: '{col_orig}' (contém 'mendes' e 'classificacao')")
        if 'guilherme' in col_norm and 'classificacao' in col_norm:
            print(f"  Encontrado: '{col_orig}' (contém 'guilherme' e 'classificacao')")
else:
    print(f"✓ Todas as colunas encontradas: {colunas_encontradas}")

# Extrai as duas colunas de avaliação
print("\nExtraindo dados dos anotadores...")
aval_mendes_raw = df['classificacao mendes'].astype(str).str.strip().str.upper()
aval_guilherme_raw = df['classificacao guilherme'].astype(str).str.strip().str.upper()

print(f"  Mendes - total de avaliações: {len(aval_mendes_raw)}")
print(f"  Guilherme - total de avaliações: {len(aval_guilherme_raw)}")

# Remove linhas onde um dos anotadores tem valor vazio/NaN
mask_validas = (aval_mendes_raw != '') & (aval_mendes_raw != 'NAN') & \
                (aval_guilherme_raw != '') & (aval_guilherme_raw != 'NAN')

aval_mendes = aval_mendes_raw[mask_validas].reset_index(drop=True)
aval_guilherme = aval_guilherme_raw[mask_validas].reset_index(drop=True)

linhas_removidas = len(aval_mendes_raw) - len(aval_mendes)
print(f"  Linhas com valores válidos: {len(aval_mendes)}")
print(f"  Linhas removidas (valores vazios/NaN): {linhas_removidas}")

# Exibe exemplos de dados
print("\nPrimeiras 5 avaliações:")
comparacao = pd.DataFrame({
    'Mendes': aval_mendes.head(),
    'Guilherme': aval_guilherme.head()
})
print(comparacao.to_string())

# Classes únicas identificadas
classes_mendes = set(aval_mendes)
classes_guilherme = set(aval_guilherme)
todas_classes = sorted(classes_mendes | classes_guilherme)

print(f"\nClasses identificadas por Mendes: {sorted(classes_mendes)}")
print(f"Classes identificadas por Guilherme: {sorted(classes_guilherme)}")
print(f"Todas as classes únicas: {todas_classes}")


Carregando arquivo de entrada...
✓ Arquivo lido com sucesso
  Total de linhas: 1415
  Total de colunas: 17
  Colunas disponíveis: ['ano', 'mes', 'dia', 'palavra_chave', 'api_key_indice', 'titulo', 'descricao', 'conteudo', 'url', 'fonte_nome', 'fonte_url', 'classificacao mendes', 'classificacao guilherme', 'classificacao marcos', 'classificacao kappa', 'categoria_geral', 'subcategoria']

Validando colunas esperadas...
✓ Todas as colunas encontradas: ['classificacao mendes', 'classificacao guilherme']

Extraindo dados dos anotadores...
  Mendes - total de avaliações: 1415
  Guilherme - total de avaliações: 1415
  Linhas com valores válidos: 1415
  Linhas removidas (valores vazios/NaN): 0

Primeiras 5 avaliações:
  Mendes Guilherme
0  MÉDIO     BAIXO
1   ALTO     MÉDIO
2  MÉDIO     BAIXO
3   ALTO     MÉDIO
4   ALTO      ALTO

Classes identificadas por Mendes: ['ALTO', 'BAIXO', 'MÉDIO']
Classes identificadas por Guilherme: ['ALTO', 'BAIXO', 'MÉDIO']
Todas as classes únicas: ['ALTO', 'BAIX

In [6]:
# Cálculo do Cohen's Kappa
# ========================
# Cohen's Kappa mede concordância entre dois anotadores
# Fórmula: κ = (p_o - p_e) / (1 - p_e)
# onde:
#   p_o = proporção de acordo observado
#   p_e = proporção de acordo esperado por acaso
# Interpretação:
#   κ < 0.20: Concordância fraca
#   0.20 ≤ κ < 0.40: Concordância razoável
#   0.40 ≤ κ < 0.60: Concordância moderada
#   0.60 ≤ κ < 0.80: Concordância substancial
#   κ ≥ 0.80: Concordância quase perfeita

print("\n" + "="*60)
print("CÁLCULO DO COHEN'S KAPPA")
print("="*60)

try:
    # Calcula o Cohen's Kappa
    kappa = cohen_kappa_score(aval_mendes, aval_guilherme)
    print(f"\nCohen's Kappa: {kappa:.4f}")
    
    # Interpretação do valor
    if kappa < 0.20:
        interpretacao = "Concordância FRACA"
    elif kappa < 0.40:
        interpretacao = "Concordância RAZOÁVEL"
    elif kappa < 0.60:
        interpretacao = "Concordância MODERADA"
    elif kappa < 0.80:
        interpretacao = "Concordância SUBSTANCIAL"
    else:
        interpretacao = "Concordância QUASE PERFEITA"
    
    print(f"Interpretação: {interpretacao}")
    
except Exception as e:
    print(f"✗ Erro ao calcular Cohen's Kappa: {e}")
    raise

# Calcula concordância simples (porcentagem de acordo direto)
print("\n" + "-"*60)
concordancia_simples = (aval_mendes == aval_guilherme).sum() / len(aval_mendes)
desacordo_simples = 1 - concordancia_simples

print(f"Concordância simples (acordo direto): {concordancia_simples*100:.2f}%")
print(f"Desacordo simples: {desacordo_simples*100:.2f}%")


CÁLCULO DO COHEN'S KAPPA

Cohen's Kappa: 0.0683
Interpretação: Concordância FRACA

------------------------------------------------------------
Concordância simples (acordo direto): 39.58%
Desacordo simples: 60.42%


In [7]:
# Matriz de confusão e relatório detalhado
# ==========================================
# Mostra a distribuição de concordâncias e discordâncias
# Permite identificar quais classificações estão causando discordância

print("\n" + "="*60)
print("MATRIZ DE CONFUSÃO")
print("="*60)

# Cria a matriz de confusão
try:
    labels = todas_classes  # todas as classes encontradas
    cm = confusion_matrix(aval_mendes, aval_guilherme, labels=labels)
    
    # Cria um DataFrame da matriz para exibição formatada
    cm_df = pd.DataFrame(
        cm,
        index=[f'Mendes: {label}' for label in labels],
        columns=[f'Guilherme: {label}' for label in labels]
    )
    
    print("\nMatriz de Confusão (linhas=Mendes, colunas=Guilherme):")
    print(cm_df.to_string())
    
    # Análise diagonal (concordâncias)
    print("\nConcordâncias por classe (diagonal da matriz):")
    for i, label in enumerate(labels):
        concordancia_classe = cm[i, i]
        total_mendes = cm[i, :].sum()
        total_guilherme = cm[:, i].sum()
        percentual = (concordancia_classe / total_mendes * 100) if total_mendes > 0 else 0
        print(f"  {label}: {concordancia_classe} concordâncias (de {total_mendes} avaliações de Mendes) = {percentual:.1f}%")
    
except Exception as e:
    print(f"✗ Erro ao gerar matriz de confusão: {e}")

# Relatório de classificação com precision, recall, f1-score
print("\n" + "-"*60)
print("Relatório de Classificação (macro-average):")
print("-"*60)

try:
    relatorio = classification_report(aval_mendes, aval_guilherme, labels=labels, zero_division=0)
    print(relatorio)
except Exception as e:
    print(f"✗ Erro ao gerar relatório: {e}")


MATRIZ DE CONFUSÃO

Matriz de Confusão (linhas=Mendes, colunas=Guilherme):
               Guilherme: ALTO  Guilherme: BAIXO  Guilherme: MÉDIO
Mendes: ALTO                87                37                84
Mendes: BAIXO               65               188               234
Mendes: MÉDIO              299               136               285

Concordâncias por classe (diagonal da matriz):
  ALTO: 87 concordâncias (de 208 avaliações de Mendes) = 41.8%
  BAIXO: 188 concordâncias (de 487 avaliações de Mendes) = 38.6%
  MÉDIO: 285 concordâncias (de 720 avaliações de Mendes) = 39.6%

------------------------------------------------------------
Relatório de Classificação (macro-average):
------------------------------------------------------------
              precision    recall  f1-score   support

        ALTO       0.19      0.42      0.26       208
       BAIXO       0.52      0.39      0.44       487
       MÉDIO       0.47      0.40      0.43       720

    accuracy                  

In [8]:
# Distribuição de classificações
# ==============================
# Mostra quantas vezes cada classe foi atribuída por cada anotador

print("\n" + "="*60)
print("DISTRIBUIÇÃO DE CLASSIFICAÇÕES")
print("="*60)

# Contagem por Mendes
mendes_contagem = aval_mendes.value_counts().sort_index()
print("\nAvaliações de Mendes:")
for classe, count in mendes_contagem.items():
    percentual = (count / len(aval_mendes)) * 100
    print(f"  {classe}: {count} ({percentual:.1f}%)")
print(f"  Total: {mendes_contagem.sum()}")

# Contagem por Guilherme
guilherme_contagem = aval_guilherme.value_counts().sort_index()
print("\nAvaliações de Guilherme:")
for classe, count in guilherme_contagem.items():
    percentual = (count / len(aval_guilherme)) * 100
    print(f"  {classe}: {count} ({percentual:.1f}%)")
print(f"  Total: {guilherme_contagem.sum()}")

# Comparação lado a lado
print("\nComparação de distribuições (Mendes vs Guilherme):")
for classe in todas_classes:
    mendes_n = (aval_mendes == classe).sum()
    guilherme_n = (aval_guilherme == classe).sum()
    mendes_pct = (mendes_n / len(aval_mendes)) * 100
    guilherme_pct = (guilherme_n / len(aval_guilherme)) * 100
    diferenca = guilherme_pct - mendes_pct
    print(f"  {classe}: Mendes={mendes_n} ({mendes_pct:.1f}%) | Guilherme={guilherme_n} ({guilherme_pct:.1f}%) | Diferença={diferenca:+.1f}%")


DISTRIBUIÇÃO DE CLASSIFICAÇÕES

Avaliações de Mendes:
  ALTO: 208 (14.7%)
  BAIXO: 487 (34.4%)
  MÉDIO: 720 (50.9%)
  Total: 1415

Avaliações de Guilherme:
  ALTO: 451 (31.9%)
  BAIXO: 361 (25.5%)
  MÉDIO: 603 (42.6%)
  Total: 1415

Comparação de distribuições (Mendes vs Guilherme):
  ALTO: Mendes=208 (14.7%) | Guilherme=451 (31.9%) | Diferença=+17.2%
  BAIXO: Mendes=487 (34.4%) | Guilherme=361 (25.5%) | Diferença=-8.9%
  MÉDIO: Mendes=720 (50.9%) | Guilherme=603 (42.6%) | Diferença=-8.3%


In [9]:
# Análise de discordâncias
# ========================
# Identifica linhas onde houve desacordo e lista quais classificações causaram conflito

print("\n" + "="*60)
print("ANÁLISE DE DISCORDÂNCIAS")
print("="*60)

# Encontra índices de desacordo
mask_desacordo = aval_mendes != aval_guilherme
linhas_desacordo = np.where(mask_desacordo)[0]
n_desacordos = len(linhas_desacordo)

print(f"\nTotal de desacordos: {n_desacordos} de {len(aval_mendes)} ({(n_desacordos/len(aval_mendes))*100:.2f}%)")
print(f"Total de acordos: {len(aval_mendes) - n_desacordos} ({((len(aval_mendes)-n_desacordos)/len(aval_mendes))*100:.2f}%)")

# Tipos de desacordo mais comuns
print("\nDesacordos mais comuns (top 10):")
desacordos_lista = []
for idx in linhas_desacordo:
    mendes = aval_mendes.iloc[idx]
    guilherme = aval_guilherme.iloc[idx]
    desacordos_lista.append((mendes, guilherme))

from collections import Counter
desacordos_contagem = Counter(desacordos_lista)
for (mendes, guilherme), count in desacordos_contagem.most_common(10):
    percentual = (count / n_desacordos) * 100
    print(f"  Mendes={mendes} → Guilherme={guilherme}: {count} vezes ({percentual:.1f}% dos desacordos)")

# Exibe alguns exemplos de linhas com desacordo (primeiras 5)
print("\nExemplos de primeiros 5 desacordos:")
for i, idx in enumerate(linhas_desacordo[:5]):
    mendes = aval_mendes.iloc[idx]
    guilherme = aval_guilherme.iloc[idx]
    print(f"  Linha {idx}: Mendes='{mendes}' vs Guilherme='{guilherme}'")


ANÁLISE DE DISCORDÂNCIAS

Total de desacordos: 855 de 1415 (60.42%)
Total de acordos: 560 (39.58%)

Desacordos mais comuns (top 10):
  Mendes=MÉDIO → Guilherme=ALTO: 299 vezes (35.0% dos desacordos)
  Mendes=BAIXO → Guilherme=MÉDIO: 234 vezes (27.4% dos desacordos)
  Mendes=MÉDIO → Guilherme=BAIXO: 136 vezes (15.9% dos desacordos)
  Mendes=ALTO → Guilherme=MÉDIO: 84 vezes (9.8% dos desacordos)
  Mendes=BAIXO → Guilherme=ALTO: 65 vezes (7.6% dos desacordos)
  Mendes=ALTO → Guilherme=BAIXO: 37 vezes (4.3% dos desacordos)

Exemplos de primeiros 5 desacordos:
  Linha 0: Mendes='MÉDIO' vs Guilherme='BAIXO'
  Linha 1: Mendes='ALTO' vs Guilherme='MÉDIO'
  Linha 2: Mendes='MÉDIO' vs Guilherme='BAIXO'
  Linha 3: Mendes='ALTO' vs Guilherme='MÉDIO'
  Linha 5: Mendes='BAIXO' vs Guilherme='MÉDIO'


In [10]:
# Salvamento de relatório em arquivo Excel
# ==========================================
# Cria um arquivo com análise completa da concordância

print("\n" + "="*60)
print("SALVAMENTO DE RELATÓRIO")
print("="*60)

try:
    with pd.ExcelWriter(EXCEL_SAIDA, engine='openpyxl') as writer:
        # Aba 1: Resumo
        resumo_data = {
            'Métrica': [
                'Cohen\'s Kappa',
                'Concordância Simples (%)',
                'Discordância Simples (%)',
                'Total de Avaliações',
                'Número de Acordos',
                'Número de Desacordos',
                'Classes Únicas'
            ],
            'Valor': [
                f"{kappa:.4f}",
                f"{concordancia_simples*100:.2f}",
                f"{desacordo_simples*100:.2f}",
                len(aval_mendes),
                len(aval_mendes) - n_desacordos,
                n_desacordos,
                len(todas_classes)
            ]
        }
        resumo_df = pd.DataFrame(resumo_data)
        resumo_df.to_excel(writer, sheet_name='Resumo', index=False)
        print(f"✓ Aba 'Resumo' criada")
        
        # Aba 2: Matriz de Confusão
        cm_df.to_excel(writer, sheet_name='Matriz Confusao')
        print(f"✓ Aba 'Matriz Confusao' criada")
        
        # Aba 3: Distribuição de Classificações
        dist_data = {}
        for classe in todas_classes:
            dist_data[classe] = [
                (aval_mendes == classe).sum(),
                (aval_guilherme == classe).sum()
            ]
        dist_df = pd.DataFrame(dist_data, index=['Mendes', 'Guilherme']).T
        dist_df.columns = ['Mendes (count)', 'Guilherme (count)']
        dist_df['Diferença'] = dist_df['Guilherme (count)'] - dist_df['Mendes (count)']
        dist_df.to_excel(writer, sheet_name='Distribuicao')
        print(f"✓ Aba 'Distribuicao' criada")
        
        # Aba 4: Desacordos
        desacordos_df_list = []
        for (mendes, guilherme), count in desacordos_contagem.items():
            desacordos_df_list.append({
                'Mendes': mendes,
                'Guilherme': guilherme,
                'Frequencia': count,
                'Percentual (%)': (count / n_desacordos) * 100
            })
        desacordos_resumo_df = pd.DataFrame(desacordos_df_list).sort_values('Frequencia', ascending=False)
        desacordos_resumo_df.to_excel(writer, sheet_name='Desacordos', index=False)
        print(f"✓ Aba 'Desacordos' criada")
        
    print(f"\n✓ Arquivo salvo com sucesso em:")
    print(f"  {EXCEL_SAIDA}")
    
except Exception as e:
    print(f"✗ Erro ao salvar arquivo: {e}")
    raise


SALVAMENTO DE RELATÓRIO
✓ Aba 'Resumo' criada
✓ Aba 'Matriz Confusao' criada
✓ Aba 'Distribuicao' criada
✓ Aba 'Desacordos' criada

✓ Arquivo salvo com sucesso em:
  c:\Users\u1174978\OneDrive - IQVIA\Documents\Geral\Importantes\MACKENZIE\5 - SEMESTRE\TCC\sistema-inteligencia-prescritiva-para-industria-farmaceutica\data\estatistica_kappa\cohen_kappa_mendes_guilherme.xlsx


In [11]:
# Resumo final
# ============

print("\n" + "="*60)
print("RESUMO FINAL DA ANÁLISE")
print("="*60)
print(f"\nCohen's Kappa (Mendes vs Guilherme): {kappa:.4f}")
print(f"Interpretação: {interpretacao}")
print(f"\nConcordância simples: {concordancia_simples*100:.2f}%")
print(f"Desacordos: {n_desacordos} ({(n_desacordos/len(aval_mendes))*100:.2f}%)")
print(f"\nArquivo de saída: {os.path.basename(EXCEL_SAIDA)}")
print("\nAnálise concluída com sucesso!")
print("="*60)


RESUMO FINAL DA ANÁLISE

Cohen's Kappa (Mendes vs Guilherme): 0.0683
Interpretação: Concordância FRACA

Concordância simples: 39.58%
Desacordos: 855 (60.42%)

Arquivo de saída: cohen_kappa_mendes_guilherme.xlsx

Análise concluída com sucesso!
